# Time Series Forecasting with BrainAutoMLForecast

Multi-stock, multi-horizon forecasting using the `brain_automl` package API.

| Step | API |
|------|-----|
| 1 | Load **full dataset** with `load_stock_forecasting_dataset(..., last_n_days=0)` |
| 2 | `model = BrainAutoMLForecast(...)` |
| 3 | `model.fit_all_stocks(df, horizons=[7, 15, 30])` — trains once per stock, evaluates at multiple horizons |
| 4 | `model.multi_stock_leaderboard()` — full results with stock, horizon, model, rmse, mae, mape |
| 5 | `model.best_model_per_stock()` — best model per stock × horizon |
| 6 | `model.overall_best_models()` — top models by mean RMSE across all stocks |

**Key Features:**
- ✅ Full dataset (no time restriction)
- ✅ Train once per stock on all data except last 30 days
- ✅ Evaluate at 7, 15, and 30-day horizons from the same trained model (efficient!)
- ✅ Per-stock leaderboard with stock name, horizon, model, library
- ✅ All AutoGluon internal models exposed (ag_DeepAR, ag_TemporalFusionTransformer, etc.)
- ✅ Univariate vs Multivariate comparison
- ✅ Visualization: RMSE by stock, horizon degradation charts

Set `include_comprehensive=True` to run classical decomposition × model sweep.  
Set `include_advanced_dl=True` to run BiLSTM / Transformer / AttentionForecaster / HybridStack (requires TensorFlow).


In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from brain_automl.data_processing import load_stock_forecasting_dataset
from brain_automl.model_zoo.time_series_ai import BrainAutoMLForecast


In [2]:
data_path = Path.cwd() / "dataset.csv"
df = load_stock_forecasting_dataset(
    data_path=data_path,
    date_col="Date",
    group_col="Stock",
    target_col="Close",
    last_n_days=0,  # Use full dataset
)

print(f"Rows: {len(df):,}  |  Stocks: {df['Stock'].nunique()}  |  Columns: {list(df.columns)}")


Rows: 18,381  |  Stocks: 5  |  Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Return', 'MA_5', 'MA_10', 'MA_20', 'Volatility', 'Stock']


In [3]:
# Multi-horizon training: trains once per stock on all data minus last 30 days
# Then evaluates at 7, 15, and 30-day horizons from the same trained model

model = BrainAutoMLForecast(
    horizon=30,  # Max horizon for training
    timestamp_column="Date",
    target_column="Close",
    item_id_column="Stock",
    include_backends=True,         # AutoGluon / StatsForecast / FLAML / Chronos
    include_comprehensive=True,   # ARIMA / LSTM / XGBoost / RF + decomposition sweep
    include_advanced_dl=True,      # BiLSTM / Transformer / Attention / HybridStack (needs TF)
)

# Trains once per stock, evaluates at multiple horizons
model.fit_all_stocks(df, horizons=[7, 15, 30])
print(model)


[22:25:56] INFO     Brain-AI run started
[22:25:56] INFO     Log file : /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output/logs/brain_automl_20260406_222556_485269.log
[22:25:56] INFO     Output dir: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output
[22:25:56] INFO     ============================================================
[22:25:56] INFO     fit_all_stocks() starting
[22:25:56] INFO       Data shape  : (18381, 12)
[22:25:56] INFO       Horizons    : [7, 15, 30]  (max=30)
[22:25:56] INFO       Target      : Close
[22:25:56] INFO       Item-ID col : 'Stock'
[22:25:56] INFO     ============================================================
[22:25:56] INFO     Stocks to process: ['BANKNIFTY', 'HDFCBANK', 'NIFTY50', 'RELIANCE', 'TCS'] (5 total)
[22:25:56] INFO     
──────────────────────────────────────────────────
[22:25:56] INFO     [1/5] Stock: BANKNIFTY
[22:2

Backends:   0%|          | 0/3 [00:00<?, ?backend/s]

[22:25:56] INFO     [START] autogluon_timeseries


[22:26:22] INFO     [ OK ] autogluon_timeseries — 26.0s | RMSE=1895.6280  MAE=1571.1698
[22:26:22] INFO     [START] statsforecast
[22:26:24] INFO     [ OK ] statsforecast — 1.5s | RMSE=1784.9661  MAE=1476.8209
[22:26:24] INFO     [START] flaml
[22:27:15] INFO     [ OK ] flaml — 51.4s | RMSE=1748.1605  MAE=1467.2402
[22:27:15] INFO     Best backend: ag_DirectTabular | RMSE=1060.0958  MAE=848.3960
[22:27:15] INFO     forecast_last_horizon complete
[22:27:15] INFO     Saved per-stock predictions to: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output/predictions/BANKNIFTY_predictions_h30.csv
[22:27:15] [comprehensive] Starting: 1 stock(s), 1 column(s), 3 decompositions, 5 models  →  up to 15 combinations
[22:27:15] [comprehensive] ── Stock: BANKNIFTY  (train=3647, test=30 rows)
[22:27:15] [comprehensive] combo 1/15 | BANKNIFTY/Close | decomp=original model=ARIMA
[22:27:15] [comprehensive]    ✓ RMSE=2205.7558  MAE=1925.9493  MAPE

Backends:   0%|          | 0/3 [00:00<?, ?backend/s]

...

[23:25:27] INFO     [START] autogluon_timeseries


[23:25:53] INFO     [ OK ] autogluon_timeseries — 26.4s | RMSE=128.4279  MAE=121.5991
[23:25:53] INFO     [START] statsforecast
[23:25:54] INFO     [ OK ] statsforecast — 0.7s | RMSE=41.4155  MAE=35.9238
[23:25:54] INFO     [START] flaml
[23:26:44] INFO     [ OK ] flaml — 49.8s | RMSE=107.0994  MAE=102.2171
[23:26:44] INFO     Best backend: ag_RecursiveTabular | RMSE=34.5617  MAE=29.7812
[23:26:44] INFO     forecast_last_horizon complete
[23:26:44] INFO     Saved per-stock predictions to: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output/predictions/RELIANCE_predictions_h30.csv
[23:26:44] [comprehensive] Starting: 1 stock(s), 1 column(s), 3 decompositions, 5 models  →  up to 15 combinations
[23:26:44] [comprehensive] ── Stock: RELIANCE  (train=3651, test=30 rows)
[23:26:44] [comprehensive] combo 1/15 | RELIANCE/Close | decomp=original model=ARIMA
[23:26:44] [comprehensive]    ✓ RMSE=35.7106  MAE=30.4268  MAPE=2.4304
[23:26:

Backends:   0%|          | 0/3 [00:00<?, ?backend/s]

[23:38:13] INFO     [START] autogluon_timeseries


[23:38:43] INFO     [ OK ] autogluon_timeseries — 29.6s | RMSE=234.1657  MAE=198.8693
[23:38:43] INFO     [START] statsforecast
[23:38:43] INFO     [ OK ] statsforecast — 0.6s | RMSE=259.7078  MAE=231.4619
[23:38:43] INFO     [START] flaml
[23:39:48] INFO     [ OK ] flaml — 64.5s | RMSE=255.1211  MAE=228.6933
[23:39:48] INFO     Best backend: ag_DirectTabular | RMSE=177.7365  MAE=140.8575
[23:39:48] INFO     forecast_last_horizon complete
[23:39:48] INFO     Saved per-stock predictions to: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output/predictions/TCS_predictions_h30.csv
[23:39:48] [comprehensive] Starting: 1 stock(s), 1 column(s), 3 decompositions, 5 models  →  up to 15 combinations
[23:39:48] [comprehensive] ── Stock: TCS  (train=3651, test=30 rows)
[23:39:48] [comprehensive] combo 1/15 | TCS/Close | decomp=original model=ARIMA
[23:39:48] [comprehensive]    ✓ RMSE=179.7535  MAE=150.9563  MAPE=3.6534
[23:39:48] [compreh

In [4]:
# Full multi-stock leaderboard with all horizons
# Shows: stock, horizon, model, library, rmse, mae, mape

full_lb = model.multi_stock_leaderboard()
print(f"Total results: {len(full_lb)} (stocks × horizons × models)")
display(full_lb.head(20))


Total results: 435 (stocks × horizons × models)


,stock,horizon,model,library,mae,mse,rmse,mape,smape,wape,mase,r2,column,target
0,HDFCBANK,7,Hybrid_ARIMA_BiLSTM_XGB,dl_multivariate,6.348257,NaN,8.467762,0.715394,NaN,NaN,NaN,NaN,NaN,Close
1,RELIANCE,7,Hybrid_ARIMA_BiLSTM_XGB,dl_univariate,7.090872,NaN,10.096055,0.582922,NaN,NaN,NaN,NaN,NaN,Close
2,HDFCBANK,7,Hybrid_ARIMA_BiLSTM_XGB,dl_univariate,10.635731,NaN,11.998385,1.205727,NaN,NaN,NaN,NaN,NaN,Close
3,RELIANCE,7,BiLSTM,dl_univariate,11.737322,NaN,15.376904,0.966799,NaN,NaN,NaN,NaN,NaN,Close
4,RELIANCE,7,XGBoostSeq,dl_multivariate,14.577267,NaN,17.116644,1.196149,NaN,NaN,NaN,NaN,NaN,Close
5,HDFCBANK,7,Transformer,dl_univariate,15.551426,NaN,18.214537,1.746103,NaN,NaN,NaN,NaN,NaN,Close
6,RELIANCE,7,XGBoostSeq,dl_univariate,15.572372,NaN,18.291878,1.274879,NaN,NaN,NaN,NaN,NaN,Close
7,HDFCBANK,7,WeightedEnsemble,autogluon_timeseries,14.083586,346.191452,18.606221,1.590334,1.612613,1.610910,1.570735,-0.990648,NaN,NaN
8,HDFCBANK,7,ETS,autogluon_internal,14.083586,346.191452,18.606221,1.590334,1.612613,1.610910,1.570735,-0.990648,NaN,NaN
9,HDFCBANK,7,arima,flaml,14.507630,361.167445,19.004406,1.638700,1.661879,1.659413,1.618028,-1.076762,NaN,NaN


In [5]:
# Setup output directory for saving results
from datetime import datetime

output_dir = Path("brain_automl_output") / "metrics"
output_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"📁 Results will be saved to: {output_dir.absolute()}")


📁 Results will be saved to: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output/metrics


In [6]:
# Best model per stock for each horizon
best_per_stock = model.best_model_per_stock()
print("Best model for each stock × horizon:")
display(best_per_stock)


Best model for each stock × horizon:


,stock,horizon,model,library,mae,mse,rmse,mape,smape,wape,mase,r2,column,target
0,BANKNIFTY,7,XGBoostSeq,dl_univariate,263.860754,1.038494e+06,306.516984,0.514165,1.433224,1.428263,1.394982,-0.735316,Close,Close
1,BANKNIFTY,15,XGBoostSeq,dl_univariate,263.860754,1.489742e+06,306.516984,0.514165,1.940851,1.933704,1.918203,-0.424435,Close,Close
2,BANKNIFTY,30,XGBoostSeq,dl_univariate,263.860754,1.123803e+06,306.516984,0.514165,1.626499,1.627388,1.608469,-0.026795,Close,Close
3,HDFCBANK,7,Hybrid_ARIMA_BiLSTM_XGB,dl_multivariate,6.348257,3.461915e+02,8.467762,0.715394,1.612613,1.610910,1.570735,-0.990648,Close,Close
4,HDFCBANK,15,Hybrid_ARIMA_BiLSTM_XGB,dl_multivariate,6.348257,2.888586e+02,8.467762,0.715394,1.563255,1.566206,1.559868,0.410659,Close,Close
5,HDFCBANK,30,Hybrid_ARIMA_BiLSTM_XGB,dl_multivariate,6.348257,1.276433e+03,8.467762,0.715394,3.389339,3.356297,3.345764,-2.284864,Close,Close
6,NIFTY50,7,Hybrid_ARIMA_BiLSTM_XGB,dl_univariate,197.876156,2.096093e+05,236.524250,0.826977,1.743643,1.737964,2.319736,-0.724712,Close,Close
7,NIFTY50,15,Hybrid_ARIMA_BiLSTM_XGB,dl_univariate,197.876156,3.123310e+05,236.524250,0.826977,2.187394,2.173471,2.939470,-0.906517,Close,Close
8,NIFTY50,30,Hybrid_ARIMA_BiLSTM_XGB,dl_univariate,197.876156,2.823424e+05,236.524250,0.826977,1.952687,1.945308,2.621952,-0.614008,Close,Close
9,RELIANCE,7,Hybrid_ARIMA_BiLSTM_XGB,dl_univariate,7.090872,7.774767e+02,10.096055,0.582922,1.831568,1.827333,1.563898,-0.083073,Close,Close


In [7]:
# Overall best models across all stocks (by mean RMSE)
overall_best = model.overall_best_models(top_n=10)
print("Top 10 models by mean RMSE across all stocks:")
display(overall_best)


Top 10 models by mean RMSE across all stocks:


,model,library,horizon,mean_rmse,std_rmse,n_stocks
0,XGBoostSeq,dl_univariate,7,177.707764,188.162390,5
1,XGBoostSeq,dl_multivariate,7,183.105569,192.762670,5
2,RecursiveTabular,autogluon_internal,7,370.749666,461.934261,5
3,AutoARIMA,statsforecast,7,384.146093,465.536830,5
4,ETS,autogluon_internal,7,387.004702,482.561149,5
5,Naive,autogluon_internal,7,392.596696,482.918611,5
6,Theta,autogluon_internal,7,407.135873,504.110767,5
7,BiLSTM,dl_univariate,7,427.734484,400.710599,5
8,BiLSTM,dl_multivariate,7,440.235322,506.081385,5
9,DirectTabular,autogluon_internal,7,445.443958,424.255300,5


In [8]:
# Horizon degradation analysis: how does RMSE change with horizon length?
# Average RMSE across all stocks for each horizon

horizon_perf = (
    full_lb.groupby(["horizon", "model", "library"])["rmse"]
    .mean()
    .reset_index()
    .sort_values("rmse")
)

# Top 5 models per horizon
top_models = []
for h in [7, 15, 30]:
    top_models.extend(
        horizon_perf[horizon_perf["horizon"] == h].head(5)["model"].tolist()
    )
top_models = list(set(top_models))

fig = px.line(
    horizon_perf[horizon_perf["model"].isin(top_models)],
    x="horizon",
    y="rmse",
    color="model",
    markers=True,
    title="Forecast Degradation: RMSE vs Horizon Length (Top Models)",
    labels={"rmse": "Mean RMSE", "horizon": "Horizon (days)"},
)
fig.show()

# Save plot as interactive HTML
plot_path = output_dir / f"horizon_degradation_plot_{timestamp}.html"
fig.write_html(plot_path)
print(f"✅ Saved horizon degradation plot to: {plot_path}")


✅ Saved horizon degradation plot to: brain_automl_output/metrics/horizon_degradation_plot_20260406_235127.html


In [9]:
# Visualize RMSE by stock and horizon

# Best model per stock-horizon combination
viz_data = best_per_stock.copy()

fig = px.bar(
    viz_data,
    x="stock",
    y="rmse",
    color="horizon",
    barmode="group",
    title="Best Model RMSE by Stock and Horizon",
    labels={"rmse": "RMSE", "stock": "Stock", "horizon": "Horizon (days)"},
    hover_data=["model", "library", "mae", "mape"],
)
fig.show()

# Save plot as interactive HTML
plot_path = output_dir / f"stock_horizon_rmse_plot_{timestamp}.html"
fig.write_html(plot_path)
print(f"✅ Saved stock×horizon RMSE plot to: {plot_path}")


✅ Saved stock×horizon RMSE plot to: brain_automl_output/metrics/stock_horizon_rmse_plot_20260406_235127.html


In [10]:
# Save all performance metrics to CSV files

# Save full leaderboard
full_lb_path = output_dir / f"full_leaderboard_{timestamp}.csv"
full_lb.to_csv(full_lb_path, index=False)
print(f"✅ Saved full leaderboard ({len(full_lb)} rows) to: {full_lb_path}")

# Save best per stock
best_per_stock_path = output_dir / f"best_per_stock_{timestamp}.csv"
best_per_stock.to_csv(best_per_stock_path, index=False)
print(f"✅ Saved best-per-stock results to: {best_per_stock_path}")

# Save overall best
overall_best_path = output_dir / f"overall_best_models_{timestamp}.csv"
overall_best.to_csv(overall_best_path, index=False)
print(f"✅ Saved overall best models to: {overall_best_path}")

# Save horizon performance analysis
horizon_perf_path = output_dir / f"horizon_degradation_{timestamp}.csv"
horizon_perf.to_csv(horizon_perf_path, index=False)
print(f"✅ Saved horizon degradation analysis to: {horizon_perf_path}")

print(f"📁 All metrics saved to: {output_dir.absolute()}")


✅ Saved full leaderboard (435 rows) to: brain_automl_output/metrics/full_leaderboard_20260406_235127.csv
✅ Saved best-per-stock results to: brain_automl_output/metrics/best_per_stock_20260406_235127.csv
✅ Saved overall best models to: brain_automl_output/metrics/overall_best_models_20260406_235127.csv
✅ Saved horizon degradation analysis to: brain_automl_output/metrics/horizon_degradation_20260406_235127.csv
📁 All metrics saved to: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output/metrics


In [11]:
# Summary of all saved files
print("\n" + "="*70)
print("📊 PERFORMANCE METRICS SAVED")
print("="*70)
print(f"\n📁 Output directory: {output_dir.absolute()}\n")
print("CSV Files:")
print(f"  • Full leaderboard: {full_lb_path.name}")
print(f"  • Best per stock: {best_per_stock_path.name}")
print(f"  • Overall best models: {overall_best_path.name}")
print(f"  • Horizon degradation: {horizon_perf_path.name}")
print(f"\nHTML Visualizations:")
print(f"  • Horizon degradation plot: horizon_degradation_plot_{timestamp}.html")
print(f"  • Stock×Horizon RMSE plot: stock_horizon_rmse_plot_{timestamp}.html")
print("\n" + "="*70)



📊 PERFORMANCE METRICS SAVED

📁 Output directory: /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output/metrics

CSV Files:
  • Full leaderboard: full_leaderboard_20260406_235127.csv
  • Best per stock: best_per_stock_20260406_235127.csv
  • Overall best models: overall_best_models_20260406_235127.csv
  • Horizon degradation: horizon_degradation_20260406_235127.csv

HTML Visualizations:
  • Horizon degradation plot: horizon_degradation_plot_20260406_235127.html
  • Stock×Horizon RMSE plot: stock_horizon_rmse_plot_20260406_235127.html



In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# STANDALONE: Load saved metrics and reproduce all charts after kernel restart
# (Filter to Close target/column only — Volatility analysis moved to a separate notebook)
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Point at the saved run ────────────────────────────────────────────────────
METRICS_DIR = Path("brain_automl_output/metrics")
RUN_TS = "20260406_235127"   # ← change if you have a newer run

full_lb = pd.read_csv(METRICS_DIR / f"full_leaderboard_{RUN_TS}.csv")
# Keep only rows related to Close (remove Volatility targets)
if 'column' in full_lb.columns:
    full_lb = full_lb[full_lb['column'].astype(str).str.lower() == 'close'].reset_index(drop=True)

# Recompute derived tables from filtered leaderboard
best_per_stock = (
    full_lb.sort_values(['stock', 'horizon', 'rmse']).
    groupby(['stock', 'horizon'], as_index=False).first()
)

overall_best = (
    full_lb.groupby('model', as_index=False)['rmse']
    .mean()
    .sort_values('rmse')
    .head(10)
)

horizon_perf = (
    full_lb.groupby(['horizon','model','library'], as_index=False)['rmse']
    .mean()
)

print(f"✅ Loaded {len(full_lb):,} leaderboard rows (Close only) | "
      f"{full_lb['stock'].nunique()} stocks | "
      f"horizons: {sorted(full_lb['horizon'].unique().tolist())}")

# ── 1. Full leaderboard ───────────────────────────────────────────────────────
print("\n── Full Leaderboard (top 20) — Close only ──")
display(full_lb.head(20))

# ── 2. Best model per stock × horizon ────────────────────────────────────────
print("\n── Best Model per Stock × Horizon (Close only) ──")
display(best_per_stock)

# ── 3. Overall best models ────────────────────────────────────────────────────
print("\n── Overall Best Models (mean RMSE across all stocks) — Close only ──")
display(overall_best)

# ── Publication-ready comparison table ───────────────────────────────────────
# Compute per-model statistics across stocks
pub_tbl = (
    full_lb.groupby(["model", "library"]).agg(
        mean_rmse=("rmse", "mean"),
        std_rmse=("rmse", "std"),
        mean_mae=("mae", "mean"),
        mean_mape=("mape", "mean"),
        n_rows=("stock", "count"),
        n_stocks=("stock", lambda s: s.nunique()),
    ).reset_index()
)

# Compute median rank of each model across stocks for RMSE (lower is better)
ranks = []
for s in sorted(full_lb['stock'].unique()):
    df_s = full_lb[full_lb['stock'] == s].copy()
    df_s = df_s.dropna(subset=['rmse'])
    df_s['rank'] = df_s.groupby('horizon')['rmse'].rank(method='min')
    ranks.append(df_s[['model', 'rank', 'stock']])
if ranks:
    rank_df = pd.concat(ranks, ignore_index=True)
    median_rank = rank_df.groupby('model')['rank'].median().reset_index().rename(columns={'rank': 'median_rank'})
    pub_tbl = pub_tbl.merge(median_rank, on='model', how='left')

pub_tbl = pub_tbl.sort_values(['mean_rmse', 'median_rank']).reset_index(drop=True)

print('\n── Publication Table (top 20 models) ──')
display(pub_tbl.head(20))

# ── 4. RMSE bar chart — best model per stock and horizon ─────────────────────
fig = px.bar(
    best_per_stock,
    x="stock",
    y="rmse",
    color="horizon",
    barmode="group",
    text="model",
    title="Best Model RMSE by Stock and Horizon (Close only)",
    labels={"rmse": "RMSE", "stock": "Stock", "horizon": "Horizon (days)"},
    hover_data=["model", "library", "mae", "mape"],
)
fig.update_traces(textposition="outside", textfont_size=9)
fig.show()

# ── 5. Horizon degradation — RMSE vs horizon length (top models) ─────────────
top_models = []
for h in [7, 15, 30]:
    top_models.extend(
        horizon_perf[horizon_perf["horizon"] == h]
        .sort_values("rmse")
        .head(5)["model"]
        .tolist()
    )
top_models = list(dict.fromkeys(top_models))  # deduplicate, preserve order

fig2 = px.line(
    horizon_perf[horizon_perf["model"].isin(top_models)].sort_values(["model", "horizon"]),
    x="horizon",
    y="rmse",
    color="model",
    markers=True,
    title="Forecast Degradation: Mean RMSE vs Horizon Length (Top Models) — Close only",
    labels={"rmse": "Mean RMSE", "horizon": "Horizon (days)"},
)
fig2.update_xaxes(tickvals=[7, 15, 30])
fig2.show()

# ── 6. Heatmap — RMSE by stock × model at each horizon ───────────────────────
for h in [7, 15, 30]:
    lb_h = full_lb[full_lb["horizon"] == h].copy()
    if lb_h.empty:
        continue
    pivot = lb_h.pivot_table(index="model", columns="stock", values="rmse", aggfunc="min")
    # Keep top 15 models by mean RMSE
    pivot = pivot.loc[pivot.mean(axis=1).sort_values().head(15).index]

    fig3 = px.imshow(
        pivot,
        color_continuous_scale="RdYlGn_r",
        title=f"RMSE Heatmap — Horizon {h}d (Close only)",
        labels=dict(x="Stock", y="Model", color="RMSE"),
        aspect="auto",
    )
    fig3.update_layout(height=450, margin=dict(l=180))
    fig3.show()

print("\n✅ All charts rendered from saved metrics (Close only).")
print("ℹ️  Volatility-specific analysis has been moved to examples/volatility_checking.ipynb")


✅ Loaded 150 leaderboard rows (Close only) | 5 stocks | horizons: [7, 15, 30]

── Full Leaderboard (top 20) — Close only ──


,stock,horizon,model,library,mae,mse,rmse,mape,smape,wape,mase,r2,column,target
0,RELIANCE,7,original/ARIMA,classical,30.426803,NaN,35.710642,2.430413,NaN,NaN,NaN,NaN,Close,NaN
1,RELIANCE,7,original/ExpSmoothing,classical,31.732232,NaN,37.878985,2.543427,NaN,NaN,NaN,NaN,Close,NaN
2,HDFCBANK,7,original/RandomForest,classical,38.602265,NaN,43.557318,4.274445,NaN,NaN,NaN,NaN,Close,NaN
3,HDFCBANK,7,original/XGBoost,classical,46.765979,NaN,51.317378,5.185904,NaN,NaN,NaN,NaN,Close,NaN
4,RELIANCE,7,original/XGBoost,classical,46.920298,NaN,57.434004,3.659946,NaN,NaN,NaN,NaN,Close,NaN
5,HDFCBANK,7,original/ExpSmoothing,classical,54.014857,NaN,58.097584,5.998009,NaN,NaN,NaN,NaN,Close,NaN
6,HDFCBANK,7,original/ARIMA,classical,57.024482,NaN,61.054448,6.334070,NaN,NaN,NaN,NaN,Close,NaN
7,RELIANCE,7,original/RandomForest,classical,50.316026,NaN,61.704480,3.924320,NaN,NaN,NaN,NaN,Close,NaN
8,RELIANCE,7,original/LSTM,classical,77.363375,NaN,86.232685,6.069063,NaN,NaN,NaN,NaN,Close,NaN
9,HDFCBANK,7,original/LSTM,classical,108.031889,NaN,109.748626,12.062002,NaN,NaN,NaN,NaN,Close,NaN



── Best Model per Stock × Horizon (Close only) ──


,stock,horizon,model,library,mae,mse,rmse,mape,smape,wape,mase,r2,column,target
0,BANKNIFTY,7,original/RandomForest,classical,1270.164705,NaN,1567.809116,2.418825,NaN,NaN,NaN,NaN,Close,None
1,BANKNIFTY,15,original/RandomForest,classical,1270.164705,NaN,1567.809116,2.418825,NaN,NaN,NaN,NaN,Close,None
2,BANKNIFTY,30,original/RandomForest,classical,1270.164705,NaN,1567.809116,2.418825,NaN,NaN,NaN,NaN,Close,None
3,HDFCBANK,7,original/RandomForest,classical,38.602265,NaN,43.557318,4.274445,NaN,NaN,NaN,NaN,Close,None
4,HDFCBANK,15,original/RandomForest,classical,38.602265,NaN,43.557318,4.274445,NaN,NaN,NaN,NaN,Close,None
5,HDFCBANK,30,original/RandomForest,classical,38.602265,NaN,43.557318,4.274445,NaN,NaN,NaN,NaN,Close,None
6,NIFTY50,7,original/XGBoost,classical,439.838150,NaN,533.753788,1.825338,NaN,NaN,NaN,NaN,Close,None
7,NIFTY50,15,original/XGBoost,classical,439.838150,NaN,533.753788,1.825338,NaN,NaN,NaN,NaN,Close,None
8,NIFTY50,30,original/XGBoost,classical,439.838150,NaN,533.753788,1.825338,NaN,NaN,NaN,NaN,Close,None
9,RELIANCE,7,original/ARIMA,classical,30.426803,NaN,35.710642,2.430413,NaN,NaN,NaN,NaN,Close,None



── Overall Best Models (mean RMSE across all stocks) — Close only ──


,model,rmse
9,original/XGBoost,490.459098
8,original/RandomForest,503.520208
6,original/ExpSmoothing,598.897553
5,original/ARIMA,642.728216
1,decomposition+model/ExpSmoothing,6972.040521
0,decomposition+model/ARIMA,6977.781852
3,decomposition+model/RandomForest,6995.223835
4,decomposition+model/XGBoost,7033.001415
2,decomposition+model/LSTM,8814.909583
7,original/LSTM,9458.892466



── Publication Table (top 20 models) ──


,model,library,mean_rmse,std_rmse,mean_mae,mean_mape,n_rows,n_stocks,median_rank
0,original/XGBoost,classical,490.459098,621.356041,403.892831,3.333585,15,5,2.0
1,original/RandomForest,classical,503.520208,601.574294,412.896337,3.240093,15,5,1.0
2,original/ExpSmoothing,classical,598.897553,792.138591,504.844031,3.501296,15,5,2.0
3,original/ARIMA,classical,642.728216,850.051122,555.459523,3.717047,15,5,4.0
4,decomposition+model/ExpSmoothing,classical,6972.040521,8792.237556,6963.744332,43.871009,15,5,6.0
5,decomposition+model/ARIMA,classical,6977.781852,8795.704711,6969.564213,43.899689,15,5,7.0
6,decomposition+model/RandomForest,classical,6995.223835,8717.673179,6985.039926,44.707581,15,5,8.0
7,decomposition+model/XGBoost,classical,7033.001415,8738.014881,7023.012136,45.004694,15,5,9.0
8,decomposition+model/LSTM,classical,8814.909583,10960.369781,8807.093945,53.650666,15,5,10.0
9,original/LSTM,classical,9458.892466,15860.986243,9012.343317,41.031526,15,5,5.0



✅ All charts rendered from saved metrics (Close only).
ℹ️  Volatility-specific analysis has been moved to examples/volatility_checking.ipynb
